# SIM Kaggle Runner

This notebook is prepared for the patched SIM repo.

Default mode is `CSG` using these Kaggle inputs:
- `/kaggle/input/datasets/manaschaiaonon/cuhk-sysu`
- `/kaggle/input/datasets/serikbayarsen/cuhk-sysu-group-reid-annotation`
- `/kaggle/input/datasets/serikbayarsen/umsot-greid`
- your uploaded `SIM_code_only` dataset

For `RoadGroup`, attach RoadGroup image and annotation inputs, then update the two variables in cell 1.

In [ ]:
# 1) Configure dataset and locate Kaggle inputs
import os
import shutil
from pathlib import Path
import yaml

DATASET_NAME = 'CSG'  # 'CSG' or 'RoadGroup'

# Update these two only if you attach RoadGroup inputs.
ROADGROUP_IMAGE_INPUT = Path('/kaggle/input/datasets/serikbayarsen/roadgroup-images')
ROADGROUP_ANN_INPUT = Path('/kaggle/input/datasets/serikbayarsen/roadgroup-annotations')
ROADGROUP_ENHANCED_INPUT = Path('/kaggle/input/datasets/serikbayarsen/roadgroup/RoadGroup_enhanced.pkl')

CSG_IMAGE_INPUT = Path('/kaggle/input/datasets/manaschaiaonon/cuhk-sysu')
CSG_ANN_INPUT = Path('/kaggle/input/datasets/serikbayarsen/cuhk-sysu-group-reid-annotation')
CSG_ANN_ENHANCED_INPUT = Path('/kaggle/input/datasets/serikbayarsen/cuhk-sysu-group-reid-annotation/group_reid_annotation_enhanced')
UMSOT_INPUT = Path('/kaggle/input/datasets/serikbayarsen/umsot-greid')
WORK_REPO = Path('/kaggle/working/SIM')

def find_sim_repo_root(root: Path) -> Path:
    candidates = [root]
    candidates.extend(p.parent for p in root.rglob('tools/train_net.py'))
    for c in candidates:
        if (c / 'fastreid').exists() and (c / 'configs').exists() and (c / 'tools').exists():
            if (c / 'fastreid/data/transforms/RoadGroup_interaction.py').exists():
                return c
    raise FileNotFoundError(f'Could not find SIM repo root under {root}')

def find_ckpt(root: Path) -> Path:
    for p in root.rglob('vit_base*.pth'):
        return p
    raise FileNotFoundError(f'Could not find ViT checkpoint under {root}')

def find_csg_image_root(root: Path) -> Path:
    candidates = [root / 'Image' / 'SSM', root / 'cuhk_sysu' / 'Image' / 'SSM']
    for c in candidates:
        if c.exists():
            return c
    for p in root.rglob('SSM'):
        if p.is_dir() and p.parent.name == 'Image':
            return p
    raise FileNotFoundError(f'Could not find CSG image root under {root}')

def find_csg_ann_root(root: Path) -> Path:
    candidates = [root / 'group_reid_annotation', root / 'cuhk_sysu' / 'group_reid_annotation']
    for c in candidates:
        if (c / 'cuhk_train.pkl').exists():
            return c
    for p in root.rglob('group_reid_annotation'):
        if (p / 'cuhk_train.pkl').exists():
            return p
    raise FileNotFoundError(f'Could not find CSG annotation root under {root}')

SIM_REPO_INPUT = None
repo_hints = [
    Path('/kaggle/input/datasets/serikbayarsen/sim-group-reid/SIM_code_only'),
    Path('/kaggle/input/datasets/serikbayarsen/sim-group-reid'),
    Path('/kaggle/input/datasets/serikbayarsen/sim-greid/SIM_code_only'),
    Path('/kaggle/input/datasets/serikbayarsen/sim-greid'),
]
for hint in repo_hints:
    if hint.exists():
        try:
            SIM_REPO_INPUT = find_sim_repo_root(hint)
            break
        except FileNotFoundError:
            pass
if SIM_REPO_INPUT is None:
    for p in Path('/kaggle/input').rglob('tools/train_net.py'):
        candidate = p.parent.parent
        if (candidate / 'fastreid/data/transforms/RoadGroup_interaction.py').exists():
            SIM_REPO_INPUT = candidate
            break
if SIM_REPO_INPUT is None:
    raise FileNotFoundError('Could not find uploaded SIM_code_only Kaggle dataset')

CKPT_PATH = find_ckpt(UMSOT_INPUT)
print('dataset:', DATASET_NAME)
print('sim_repo_input:', SIM_REPO_INPUT)
print('checkpoint:', CKPT_PATH)


In [ ]:
# 2) Copy repo to writable workspace and patch local paths
if WORK_REPO.exists():
    shutil.rmtree(WORK_REPO)
shutil.copytree(SIM_REPO_INPUT, WORK_REPO)

if DATASET_NAME == 'CSG':
    image_root = find_csg_image_root(CSG_IMAGE_INPUT)
    ann_root = find_csg_ann_root(CSG_ANN_INPUT)
    dataset_root = WORK_REPO / 'datasets' / 'cuhk_sysu'
    dataset_root.mkdir(parents=True, exist_ok=True)
    image_link = dataset_root / 'Image' / 'SSM'
    image_link.parent.mkdir(parents=True, exist_ok=True)
    ann_link = dataset_root / 'group_reid_annotation'
    if image_link.exists() or image_link.is_symlink():
        image_link.unlink()
    if ann_link.exists() or ann_link.is_symlink():
        ann_link.unlink()
    os.symlink(image_root, image_link)
    os.symlink(ann_root, ann_link)
    csg_enhanced_root = CSG_ANN_ENHANCED_INPUT if CSG_ANN_ENHANCED_INPUT.exists() else ann_link
    local_paths = {
        'MODEL': {'BACKBONE': {'PRETRAIN_PATH': str(CKPT_PATH)}},
        'DATASETS': {
            'ROOT': str(WORK_REPO / 'datasets'),
            'CSG_ROOT': str(dataset_root),
            'CSG_IMAGE_ROOT': str(image_link),
            'CSG_LABEL_ROOT': str(ann_link),
            'CSG_ENHANCED_ROOT': str(csg_enhanced_root),
            'ROADGROUP_ROOT': str(WORK_REPO / 'datasets' / 'RoadGroup'),
            'ROADGROUP_IMAGE_ROOT': str(WORK_REPO / 'datasets' / 'RoadGroup' / 'Road_Group' / 'group'),
            'ROADGROUP_ANNOTATION_ROOT': str(WORK_REPO / 'datasets' / 'RoadGroup' / 'Road_Group_Annotations'),
            'ROADGROUP_ENHANCED_PATH': str(WORK_REPO / 'datasets' / 'RoadGroup' / 'Road_Group_Annotations' / 'RoadGroup_enhanced.pkl'),
        },
    }
elif DATASET_NAME == 'RoadGroup':
    road_image_root = ROADGROUP_IMAGE_INPUT
    road_ann_root = ROADGROUP_ANN_INPUT
    if not road_image_root.exists() or not road_ann_root.exists():
        raise FileNotFoundError('Attach RoadGroup image and annotation datasets, then update ROADGROUP_* variables in cell 1')
    dataset_root = WORK_REPO / 'datasets' / 'RoadGroup'
    image_link = dataset_root / 'Road_Group' / 'group'
    ann_link = dataset_root / 'Road_Group_Annotations'
    image_link.parent.mkdir(parents=True, exist_ok=True)
    ann_link.parent.mkdir(parents=True, exist_ok=True)
    if image_link.exists() or image_link.is_symlink():
        image_link.unlink()
    if ann_link.exists() or ann_link.is_symlink():
        ann_link.unlink()
    os.symlink(road_image_root, image_link)
    os.symlink(road_ann_root, ann_link)
    road_enhanced_path = ROADGROUP_ENHANCED_INPUT if ROADGROUP_ENHANCED_INPUT.exists() else (ann_link / 'RoadGroup_enhanced.pkl')
    local_paths = {
        'MODEL': {'BACKBONE': {'PRETRAIN_PATH': str(CKPT_PATH)}},
        'DATASETS': {
            'ROOT': str(WORK_REPO / 'datasets'),
            'CSG_ROOT': str(WORK_REPO / 'datasets' / 'cuhk_sysu'),
            'CSG_IMAGE_ROOT': str(WORK_REPO / 'datasets' / 'cuhk_sysu' / 'Image' / 'SSM'),
            'CSG_LABEL_ROOT': str(WORK_REPO / 'datasets' / 'cuhk_sysu' / 'group_reid_annotation'),
            'CSG_ENHANCED_ROOT': str(CSG_ANN_ENHANCED_INPUT if CSG_ANN_ENHANCED_INPUT.exists() else (WORK_REPO / 'datasets' / 'cuhk_sysu' / 'group_reid_annotation')),
            'ROADGROUP_ROOT': str(dataset_root),
            'ROADGROUP_IMAGE_ROOT': str(image_link),
            'ROADGROUP_ANNOTATION_ROOT': str(ann_link),
            'ROADGROUP_ENHANCED_PATH': str(road_enhanced_path),
        },
    }
else:
    raise ValueError(f'Unsupported DATASET_NAME: {DATASET_NAME}')

with open(WORK_REPO / 'configs' / 'local_paths.yml', 'w') as f:
    yaml.safe_dump(local_paths, f, sort_keys=False)

cfg_path = WORK_REPO / 'configs' / DATASET_NAME / 'bagtricks_gvit_local.yml'
if not cfg_path.exists():
    cfg_path.write_text('_BASE_: bagtricks_gvit.yml\n\nOUTPUT_DIR: logs/' + DATASET_NAME + '/local_sim\n')
text = cfg_path.read_text()
if 'OUTPUT_DIR:' in text:
    import re
    text = re.sub(r'OUTPUT_DIR:\s*.*', 'OUTPUT_DIR: /kaggle/working/sim_logs', text)
else:
    text += '\nOUTPUT_DIR: /kaggle/working/sim_logs\n'
cfg_path.write_text(text)

print('work_repo:', WORK_REPO)
print('dataset_root:', dataset_root)
print('cfg_path:', cfg_path)
print((WORK_REPO / 'configs' / 'local_paths.yml').read_text())


In [ ]:
# 3) Install dependencies
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(WORK_REPO / 'requirements.txt')], check=True)


In [ ]:
# 4) Sanity checks
%cd /kaggle/working/SIM

if DATASET_NAME == 'CSG':
    from fastreid.data.datasets.CSG import CSG
    ds = CSG(
        dataset_root=str(WORK_REPO / 'datasets' / 'cuhk_sysu'),
        image_root=str(WORK_REPO / 'datasets' / 'cuhk_sysu' / 'Image' / 'SSM'),
        label_root=str(WORK_REPO / 'datasets' / 'cuhk_sysu' / 'group_reid_annotation'),
        enhanced_root=str(CSG_ANN_ENHANCED_INPUT if CSG_ANN_ENHANCED_INPUT.exists() else (WORK_REPO / 'datasets' / 'cuhk_sysu' / 'group_reid_annotation')),
    )
else:
    from fastreid.data.datasets.RoadGroup import RoadGroup
    ds = RoadGroup(
        dataset_root=str(WORK_REPO / 'datasets' / 'RoadGroup'),
        image_root=str(WORK_REPO / 'datasets' / 'RoadGroup' / 'Road_Group' / 'group'),
        annotation_root=str(WORK_REPO / 'datasets' / 'RoadGroup' / 'Road_Group_Annotations'),
        enhanced_path=str(ROADGROUP_ENHANCED_INPUT if ROADGROUP_ENHANCED_INPUT.exists() else (WORK_REPO / 'datasets' / 'RoadGroup' / 'Road_Group_Annotations' / 'RoadGroup_enhanced.pkl')),
    )

print('train/query/gallery:', len(ds.train), len(ds.query), len(ds.gallery))
print('sample:', ds.train[0])

from fastreid.config import get_cfg
from fastreid.modeling.meta_arch import build_model

cfg = get_cfg()
cfg.merge_from_file(f'configs/{DATASET_NAME}/bagtricks_gvit_local.yml')
cfg.merge_from_file('configs/local_paths.yml')
cfg.merge_from_list(['MODEL.DEVICE', 'cpu'])
cfg.freeze()
model = build_model(cfg)
print(type(model))


In [ ]:
# 5) Optional: build RoadGroup interaction-enhanced annotations
%cd /kaggle/working/SIM

if DATASET_NAME == 'RoadGroup':
    enhanced_path = ROADGROUP_ENHANCED_INPUT if ROADGROUP_ENHANCED_INPUT.exists() else (WORK_REPO / 'datasets' / 'RoadGroup' / 'Road_Group_Annotations' / 'RoadGroup_enhanced.pkl')
    if enhanced_path.exists():
        print('already exists:', enhanced_path)
    else:
        !python -u fastreid/data/transforms/RoadGroup_interaction.py \
          --image-root /kaggle/working/SIM/datasets/RoadGroup/Road_Group/group \
          --annotation-root /kaggle/working/SIM/datasets/RoadGroup/Road_Group_Annotations
else:
    print('Skip. Only needed for RoadGroup.')


In [ ]:
# 6) Optional smoke test
%cd /kaggle/working/SIM

!python -u tools/train_net.py \
  --dataset {DATASET_NAME} \
  MODEL.DEVICE cuda \
  SOLVER.MAX_EPOCH 1 \
  TEST.EVAL_PERIOD 1 \
  SOLVER.CHECKPOINT_PERIOD 1 \
  DATALOADER.NUM_WORKERS 2


In [ ]:
# 7) Clean output dir before full run
out = Path('/kaggle/working/sim_logs')
if out.exists():
    shutil.rmtree(out)
print('cleaned', out)


In [ ]:
# 8) Full training on Kaggle T4 x2
%cd /kaggle/working/SIM

!python -u tools/train_net.py \
  --dataset {DATASET_NAME} \
  --num-gpus 2 \
  DATALOADER.NUM_WORKERS 2 \
  SOLVER.IMS_PER_BATCH 16 \
  TEST.EVAL_PERIOD 10 \
  SOLVER.CHECKPOINT_PERIOD 10


In [ ]:
# 9) Paper-style final evaluation on the best checkpoint
%cd /kaggle/working/SIM

!python -u tools/train_net.py \
  --dataset {DATASET_NAME} \
  --eval-only \
  MODEL.DEVICE cuda \
  MODEL.WEIGHTS /kaggle/working/sim_logs/model_best.pth


In [ ]:
# 10) Optional: compare against the final checkpoint
%cd /kaggle/working/SIM

!python -u tools/train_net.py \
  --dataset {DATASET_NAME} \
  --eval-only \
  MODEL.DEVICE cuda \
  MODEL.WEIGHTS /kaggle/working/sim_logs/model_final.pth


In [ ]:
# 11) Retrieval visualization on the test set using the best checkpoint
%cd /kaggle/working/SIM

!python -u tools/eval_test_set.py \
  --config-file configs/{DATASET_NAME}/bagtricks_gvit_local.yml \
  --path-config configs/local_paths.yml \
  --dataset-name {DATASET_NAME} \
  --weights /kaggle/working/sim_logs/model_best.pth \
  --output-dir /kaggle/working/sim_retrieval \
  --num-vis 12 \
  --vis-topk 3


In [ ]:
# 12) Preview retrieval visualization and metrics
from pathlib import Path
from IPython.display import display, Image as IPyImage
import json

dataset_dir = Path('/kaggle/working/sim_retrieval') / DATASET_NAME
grid_path = dataset_dir / 'retrieval_grid.jpg'
metrics_path = dataset_dir / 'metrics.json'
manifest_path = dataset_dir / 'retrieval_manifest.json'

print('grid:', grid_path)
print('metrics:', metrics_path)
print('manifest:', manifest_path)

if metrics_path.exists():
    print(json.dumps(json.loads(metrics_path.read_text()), indent=2))

if grid_path.exists():
    display(IPyImage(filename=str(grid_path)))
else:
    print('retrieval grid not found')
